In [1]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path('datasets')

finance_df = pd.read_json(DATA_DIR / 'finance.jsonl', lines=True)
medicine_df = pd.read_json(DATA_DIR / 'medicine.jsonl', lines=True)
open_qa_df = pd.read_json(DATA_DIR / 'open_qa.jsonl', lines=True)
reddit_eli5_df = pd.read_json(DATA_DIR / 'reddit_eli5.jsonl', lines=True)
wiki_csai_df = pd.read_json(DATA_DIR / 'wiki_csai.jsonl', lines=True)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

def df_to_text_label(df, source_name):
    human = (
        df[['human_answers']]
        .explode('human_answers')
        .dropna()
        .rename(columns={'human_answers': 'text'})
    )
    human['label'] = 1

    ai = (
        df[['chatgpt_answers']]
        .explode('chatgpt_answers')
        .dropna()
        .rename(columns={'chatgpt_answers': 'text'})
    )
    ai['label'] = 0

    human['source'] = source_name
    ai['source'] = source_name

    return pd.concat([human, ai], ignore_index=True)

datasets = {
    "finance": df_to_text_label(finance_df, "finance"),
    "medicine": df_to_text_label(medicine_df, "medicine"),
    "open_qa": df_to_text_label(open_qa_df, "open_qa"),
    "reddit_eli5": df_to_text_label(reddit_eli5_df, "reddit_eli5"),
    "wiki_csai": df_to_text_label(wiki_csai_df, "wiki_csai"),
}

for name, df in datasets.items():
    print(name, df.shape)

finance (8436, 3)
medicine (2585, 3)
open_qa (4748, 3)
reddit_eli5 (67996, 3)
wiki_csai (1684, 3)


In [ ]:
results = []

for test_name, test_df in datasets.items():
    train_domain = [df for name, df in datasets.items() if name != test_name]
    train_df = pd.concat(train_domain, ignore_index=True)

    X_train, y_train = train_df['text'], train_df['label']
    X_test, y_test = test_df['text'], test_df['label']

    clf = Pipeline([
        ("tfidf", TfidfVectorizer(
            max_features=200000,
            ngram_range=(1, 2),
            min_df=2
        )),
        ("logreg", LogisticRegression(
            max_iter=1000,
            n_jobs=-1,
        )),
    ])

    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results.append({"test_domain": test_name, "accuracy": acc})

    print(f"=== Leave out {test_name} (train on the other 4) ===")
    print(f"Accuracy: {acc:.4f}")
    print()

avg_acc = sum(r["accuracy"] for r in results) / len(results)
print("Average accuracy over all leave-one-domain-out runs:", avg_acc)
results

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =       200001     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  6.93147D-01    |proj g|=  2.09140D-01


 This problem is unconstrained.



At iterate   50    f=  1.61129D-01    |proj g|=  4.53075D-04

           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tnf  Tnint  Skip  Nact     Projg        F
*****     52     57      1     0     0   5.338D-05   1.611D-01
  F =  0.16109781586977917     

CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL            
=== Leave out finance (train on the other 4) ===
Accuracy: 0.9339



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


...


           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tnf  Tnint  Skip  Nact     Projg        F
*****     35     38      1     0     0   4.639D-05   1.467D-01
  F =  0.14673004430140407     

CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL            
=== Leave out open_qa (train on the other 4) ===
Accuracy: 0.6763



[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.


RUNNING THE L-BFGS-B CODE

           * * *

Machine precision = 2.220D-16
 N =       200001     M =           10

At X0         0 variables are exactly at the bounds

At iterate    0    f=  6.93147D-01    |proj g|=  8.68905D-02


 This problem is unconstrained.



           * * *

Tit   = total number of iterations
Tnf   = total number of function evaluations
Tnint = total number of segments explored during Cauchy searches
Skip  = number of BFGS updates skipped
Nact  = number of active bounds at final generalized Cauchy point
Projg = norm of the final projected gradient
F     = final function value

           * * *

   N    Tit     Tnf  Tnint  Skip  Nact     Projg        F
*****     17     24      1     0     0   8.101D-05   2.736D-01
  F =  0.27359439534093744     

CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL            
=== Leave out reddit_eli5 (train on the other 4) ===
Accuracy: 0.9067



### Interpreting Results

From the above cell we can see that the logistic regression model trained is able to generalize across domains. However we should note that when the `open_qa` and `wiki_csai` datasets are left out of training, the accuracy score is observably worse. Additionally, the largest bulk of our dataset (`reddit_eli5`) being left out still resulted in a high accuracy score.

As such, we conclude that the most important datasets that contain features that can identify chatgpt generated text are the `open_qa` and `wiki_csai` datasets.